# Exploration du flux v2 des stations (issue #22)
Source : flux v2 des prix des carburants 

In [ ]:
import sys

import pandas as pd

sys.path.append("..")

from src.cleaning import charger_stations, lire_flux_v2

In [ ]:
stations_brutes = lire_flux_v2("../data/raw/flux_v2.csv")

In [ ]:
print(stations_brutes.shape)
print(stations_brutes.columns.tolist())
stations_brutes.head()

In [ ]:
print(stations_brutes["pop"].value_counts(dropna=False))
print(stations_brutes["id"].duplicated().sum()) 

In [ ]:
print(stations_brutes["Région"].value_counts(dropna=False))

sans_region = stations_brutes[stations_brutes["Région"].isna()]
sans_region[["id", "Code postal", "Ville", "code_departement", "pop"]]

In [ ]:
stations_idf = charger_stations("../data/raw/flux_v2.csv")
print(stations_idf.shape)

stations_region = stations_brutes[stations_brutes["Région"] == "Île-de-France"]
assert len(stations_idf) == len(stations_region)

In [ ]:
latitude_degres = stations_idf["latitude"].astype(float) / 100000
longitude_degres = stations_idf["longitude"].astype(float) / 100000
print(latitude_degres.min(), latitude_degres.max())   
print(longitude_degres.min(), longitude_degres.max())  

In [ ]:
print(stations_idf["Code postal"].str.len().value_counts())


prefixe_postal = stations_idf["Code postal"].str[:2]
incoherentes = stations_idf[prefixe_postal != stations_idf["code_departement"]]
print(len(incoherentes))
incoherentes[["id", "Ville", "Code postal", "code_departement"]]

In [ ]:
print((stations_idf["pop"] == "A").sum())  

In [ ]:
historique_2021 = pd.read_xml(
    "../data/raw/PrixCarburants_annuel_2021.xml",
    xpath=".//pdv",
    attrs_only=True,
    parser="etree",
    encoding="ISO-8859-1",
    dtype=str,
)
print(historique_2021.shape)

In [ ]:
print(historique_2021["id"].duplicated().sum())  # on attend 0

dans_le_flux = historique_2021["id"].isin(stations_brutes["id"])
absents_du_flux = historique_2021[~dans_le_flux]

print(len(historique_2021), len(absents_du_flux))
print(absents_du_flux["pop"].value_counts())

In [ ]:
from src.cleaning import DEPARTEMENTS_IDF, charger_stations, lire_flux_v2

In [ ]:
prefixe_cp = historique_2021["cp"].str[:2]
historique_idf = historique_2021[prefixe_cp.isin(DEPARTEMENTS_IDF)]
absents_idf = historique_idf[~historique_idf["id"].isin(stations_brutes["id"])]

print(len(historique_idf), len(absents_idf))
print(absents_idf["pop"].value_counts())


historique_2021[historique_2021["pop"] == "N"]